In [1]:
!pip install -q transformers datasets peft accelerate sentencepiece evaluate pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading base model...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model = base_model.to(device)

print("Model loaded successfully!")
print("Model:", MODEL_NAME)
print("Device:", device)

Loading tokenizer...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!
Model: google/flan-t5-small
Device: cuda


In [4]:
def generate_response(prompt, model=base_model):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


test_prompt = """
You are a market research analyst.
Analyze the main factors that influence Gen Z consumers
when choosing a mobile banking provider.
"""

response = generate_response(test_prompt)

print("PROMPT:")
print(test_prompt)

print("\nBASE MODEL RESPONSE:")
print(response)

PROMPT:

You are a market research analyst.
Analyze the main factors that influence Gen Z consumers
when choosing a mobile banking provider.


BASE MODEL RESPONSE:
Gen Z consumers are more likely to use mobile banking services.


In [5]:
import pandas as pd

data = [
    {
        "category": "Consumer Behavior",
        "prompt": "What factors influence Gen Z consumers when choosing a mobile banking provider?",
        "response": "Gen Z consumers are influenced by mobile app usability, low fees, digital payment integration, security, customer service accessibility, financial education tools, brand reputation, and personalized features."
    },
    {
        "category": "Consumer Behavior",
        "prompt": "What factors influence millennials when selecting a meal delivery service?",
        "response": "Millennials often consider delivery speed, convenience, pricing, restaurant selection, app usability, promotions, sustainability, customer reviews, and loyalty rewards when choosing a meal delivery service."
    },
    {
        "category": "Consumer Behavior",
        "prompt": "What drives customer loyalty in subscription streaming services?",
        "response": "Customer loyalty in streaming services is influenced by content quality, exclusive programming, price, ease of use, personalization, recommendation quality, device compatibility, and consistent platform performance."
    },
    {
        "category": "Market Trends",
        "prompt": "What market trends are currently influencing electric vehicle adoption?",
        "response": "Electric vehicle adoption is influenced by battery improvements, charging infrastructure expansion, government incentives, fuel prices, environmental awareness, vehicle affordability, manufacturer investment, and consumer concerns about driving range."
    },
    {
        "category": "Market Trends",
        "prompt": "What major trends are shaping the online grocery delivery market?",
        "response": "Major trends include increased demand for convenience, subscription delivery models, faster fulfillment, mobile ordering, personalized promotions, retailer partnerships, AI-driven recommendations, and growing competition among delivery platforms."
    },
    {
        "category": "Market Trends",
        "prompt": "What trends are affecting consumer demand for wearable fitness technology?",
        "response": "Demand is being shaped by health awareness, remote wellness monitoring, improved sensors, integration with smartphones, personalization, sleep tracking, affordability, and interest in preventive healthcare."
    },
    {
        "category": "Competitor Analysis",
        "prompt": "How should a new coffee shop analyze its local competitors?",
        "response": "The business should compare competitor pricing, menu variety, product quality, customer reviews, location, atmosphere, promotions, loyalty programs, service quality, operating hours, and target customer segments."
    },
    {
        "category": "Competitor Analysis",
        "prompt": "How can a new fitness app compare itself with established competitors?",
        "response": "The company should evaluate competitor features, subscription pricing, user experience, personalization, workout variety, customer reviews, retention strategies, integrations, brand positioning, and unique value propositions."
    },
    {
        "category": "Competitor Analysis",
        "prompt": "What factors should be included in a competitor analysis for a clothing brand?",
        "response": "A clothing brand should assess competitor pricing, product quality, style, target demographics, distribution channels, social media presence, brand reputation, promotions, customer reviews, sustainability practices, and market positioning."
    },
    {
        "category": "Customer Satisfaction",
        "prompt": "What factors most influence customer satisfaction in restaurants?",
        "response": "Customer satisfaction is influenced by food quality, service speed, staff professionalism, cleanliness, pricing, atmosphere, order accuracy, menu variety, convenience, and how complaints are handled."
    },
    {
        "category": "Customer Satisfaction",
        "prompt": "How can a software company measure customer satisfaction?",
        "response": "A software company can use customer satisfaction surveys, Net Promoter Score, support ticket analysis, churn rates, product reviews, customer interviews, usage data, renewal rates, and customer feedback trends."
    },
    {
        "category": "Customer Satisfaction",
        "prompt": "What causes dissatisfaction among online retail customers?",
        "response": "Common causes include late delivery, inaccurate product descriptions, poor product quality, difficult returns, hidden fees, unresponsive customer service, website problems, payment issues, and poor communication."
    },
    {
        "category": "Marketing Strategy",
        "prompt": "How can a new fitness brand attract millennial customers?",
        "response": "The brand can use social media marketing, influencer partnerships, personalized fitness content, flexible membership options, community engagement, referral programs, wellness education, and mobile-first experiences."
    },
    {
        "category": "Marketing Strategy",
        "prompt": "What marketing strategies can help a local restaurant attract new customers?",
        "response": "Effective strategies include social media promotions, local partnerships, loyalty programs, online reviews, special events, targeted advertising, delivery partnerships, email marketing, and referral incentives."
    },
    {
        "category": "Marketing Strategy",
        "prompt": "How can an online retailer improve customer acquisition?",
        "response": "The retailer can improve acquisition through search advertising, social media campaigns, influencer marketing, referral programs, email campaigns, search engine optimization, personalized promotions, and improved landing pages."
    },
    {
        "category": "Demographic Analysis",
        "prompt": "How should a company compare purchasing behavior across age groups?",
        "response": "The company should compare purchase frequency, average spending, preferred channels, product preferences, price sensitivity, brand loyalty, digital engagement, and responses to promotions across age groups."
    },
    {
        "category": "Demographic Analysis",
        "prompt": "What demographic factors should be considered when launching a new health product?",
        "response": "Key demographic factors include age, income, gender, location, education, occupation, household size, health needs, cultural preferences, and access to healthcare."
    },
    {
        "category": "Demographic Analysis",
        "prompt": "How can income level affect consumer purchasing decisions?",
        "response": "Income level can influence price sensitivity, product quality expectations, brand preferences, purchase frequency, financing needs, discretionary spending, and willingness to pay for premium features."
    },
    {
        "category": "Regional Analysis",
        "prompt": "How should a company evaluate regional differences before expanding into a new market?",
        "response": "The company should analyze local income levels, cultural preferences, regulations, competition, infrastructure, consumer behavior, pricing expectations, economic conditions, demographics, and distribution channels."
    },
    {
        "category": "Regional Analysis",
        "prompt": "What factors should be considered when comparing urban and rural markets?",
        "response": "Important factors include population density, income, transportation access, internet availability, retail access, customer needs, pricing sensitivity, competition, logistics, and preferred shopping channels."
    },
    {
        "category": "Regional Analysis",
        "prompt": "How can cultural differences affect international market research?",
        "response": "Cultural differences can influence language, values, purchasing habits, communication styles, brand perception, product preferences, survey responses, and attitudes toward advertising."
    },
    {
        "category": "Predictive Market Research",
        "prompt": "How can a company predict future demand for a new product?",
        "response": "The company can analyze historical sales, market growth rates, customer surveys, search trends, competitor activity, economic indicators, demographic changes, seasonality, and pilot launch results."
    },
    {
        "category": "Predictive Market Research",
        "prompt": "What data can help predict customer churn?",
        "response": "Useful data includes product usage, complaint frequency, support interactions, payment behavior, contract renewal history, satisfaction scores, engagement trends, service disruptions, and changes in purchasing behavior."
    },
    {
        "category": "Predictive Market Research",
        "prompt": "How can retailers forecast seasonal demand?",
        "response": "Retailers can use historical sales, holiday calendars, weather patterns, promotional schedules, economic trends, inventory data, online search trends, and previous seasonal performance."
    }
]

df = pd.DataFrame(data)

print("Dataset created successfully!")
print("Total examples:", len(df))
display(df.head())

Dataset created successfully!
Total examples: 24


,category,prompt,response
0,Consumer Behavior,What factors influence Gen Z consumers when ch...,Gen Z consumers are influenced by mobile app u...
1,Consumer Behavior,What factors influence millennials when select...,"Millennials often consider delivery speed, con..."
2,Consumer Behavior,What drives customer loyalty in subscription s...,Customer loyalty in streaming services is infl...
3,Market Trends,What market trends are currently influencing e...,Electric vehicle adoption is influenced by bat...
4,Market Trends,What major trends are shaping the online groce...,Major trends include increased demand for conv...


In [6]:
df.to_csv("market_research_dataset.csv", index=False)

print("Saved market_research_dataset.csv")

Saved market_research_dataset.csv


In [7]:
from sklearn.model_selection import train_test_split

# First split: training vs temporary set
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["category"]
)

# Second split: temporary set into validation and test
validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))
print("Held-out test examples:", len(test_df))

Training examples: 16
Validation examples: 4
Held-out test examples: 4


In [8]:
train_df.to_csv("train.csv", index=False)
validation_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Saved:")
print("- train.csv")
print("- validation.csv")
print("- test.csv")

Saved:
- train.csv
- validation.csv
- test.csv


In [9]:
print("HELD-OUT TEST DATA")
display(test_df)

HELD-OUT TEST DATA


,category,prompt,response
18,Regional Analysis,How should a company evaluate regional differe...,The company should analyze local income levels...
8,Competitor Analysis,What factors should be included in a competito...,A clothing brand should assess competitor pric...
0,Consumer Behavior,What factors influence Gen Z consumers when ch...,Gen Z consumers are influenced by mobile app u...
5,Market Trends,What trends are affecting consumer demand for ...,"Demand is being shaped by health awareness, re..."


In [10]:
from datasets import load_dataset

data_files = {
    "train": "train.csv",
    "validation": "validation.csv",
    "test": "test.csv"
}

dataset = load_dataset("csv", data_files=data_files)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['category', 'prompt', 'response'],
        num_rows: 16
    })
    validation: Dataset({
        features: ['category', 'prompt', 'response'],
        num_rows: 4
    })
    test: Dataset({
        features: ['category', 'prompt', 'response'],
        num_rows: 4
    })
})


In [11]:
def format_prompt(example):
    return (
        "You are a market research analyst. "
        "Provide a clear, relevant, and actionable market research response.\n\n"
        f"Question: {example['prompt']}\n"
        "Answer:"
    )

print(format_prompt(train_df.iloc[0]))

You are a market research analyst. Provide a clear, relevant, and actionable market research response.

Question: What market trends are currently influencing electric vehicle adoption?
Answer:


In [12]:
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 128

def preprocess_function(examples):

    inputs = [
        "You are a market research analyst. "
        "Provide a clear, relevant, and actionable market research response.\n\n"
        f"Question: {prompt}\n"
        "Answer:"
        for prompt in examples["prompt"]
    ]

    targets = examples["response"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [13]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 16
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4
    })
})


In [20]:
import torch
!pip install --upgrade torchao

from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSeq2SeqLM

training_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v"]
)

training_model = get_peft_model(
    training_model,
    lora_config
)

training_model = training_model.to(device)

training_model.print_trainable_parameters()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 108.3 MB/s eta 0:00:00


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451


In [15]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [16]:
!pip install -q transformers datasets peft accelerate sentencepiece evaluate pandas scikit-learn

In [17]:
import torch
import transformers
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

PyTorch: 2.11.0+cu128
Transformers: 5.13.1
PEFT: 0.19.1
Accelerate: 1.14.0


In [23]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSeq2SeqLM

training_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v"]
)

training_model = get_peft_model(
    training_model,
    lora_config
)

training_model = training_model.to(device)

training_model.print_trainable_parameters()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451


In [30]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=training_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer created successfully.")

NameError: name 'data_collator' is not defined

In [25]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./sba928_model",

    learning_rate=5e-4,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=10,

    weight_decay=0.01,

    logging_strategy="steps",
    logging_steps=1,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=2,

    predict_with_generate=True,

    fp16=torch.cuda.is_available(),

    report_to="none",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False
)

print(training_args)

Seq2SeqTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOC

In [32]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=training_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer created successfully.")

NameError: name 'data_collator' is not defined

In [33]:
train_result = trainer.train()

NameError: name 'trainer' is not defined

In [34]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=training_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer created successfully.")

NameError: name 'data_collator' is not defined

In [35]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=training_model
)

print("Data collator ready.")

Data collator ready.


In [36]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=training_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer created successfully.")

Trainer created successfully.


In [37]:
print(type(trainer))

<class 'transformers.trainer_seq2seq.Seq2SeqTrainer'>


In [38]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan
4,0.000000,nan
5,0.000000,nan
6,0.000000,nan
7,0.000000,nan
8,0.000000,nan
9,0.000000,nan
10,0.000000,nan


In [39]:
print("TRAINING COMPLETE")

final_eval = trainer.evaluate()

print(final_eval)

TRAINING COMPLETE


Training Loss,Validation Loss,Epoch
0.000000,nan,10


{'eval_loss': nan}


In [40]:
ADAPTER_PATH = "sba928_market_research_adapter"

training_model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("Adapter saved successfully.")
print("Location:", ADAPTER_PATH)

Adapter saved successfully.
Location: sba928_market_research_adapter


In [41]:
training_logs = pd.DataFrame(trainer.state.log_history)

training_logs.to_csv("training_log.csv", index=False)

print("Training log saved.")
display(training_logs)

Training log saved.


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,0.0,NaN,0.000500,0.25,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,NaN,0.000487,0.50,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,NaN,0.000475,0.75,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0,NaN,0.000463,1.00,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,1.00,4,NaN,0.0781,51.212,12.803,NaN,NaN,NaN,NaN,NaN
5,0.0,NaN,0.000450,1.25,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.0,NaN,0.000438,1.50,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.0,NaN,0.000425,1.75,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.0,NaN,0.000412,2.00,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,2.00,8,NaN,0.1345,29.749,7.437,NaN,NaN,NaN,NaN,NaN


In [42]:
base_results = []

for _, row in test_df.iterrows():
    prompt_text = (
        "You are a market research analyst. "
        "Provide a clear, relevant, and actionable market research response.\n\n"
        f"Question: {row['prompt']}\n"
        "Answer:"
    )

    response = generate_response(prompt_text, model=base_model)

    base_results.append({
        "category": row["category"],
        "prompt": row["prompt"],
        "expected_response": row["response"],
        "base_model_response": response
    })

base_outputs_df = pd.DataFrame(base_results)

base_outputs_df.to_csv("base_model_outputs.csv", index=False)

display(base_outputs_df)

,category,prompt,expected_response,base_model_response
0,Regional Analysis,How should a company evaluate regional differe...,The company should analyze local income levels...,Identify regional differences.
1,Competitor Analysis,What factors should be included in a competito...,A clothing brand should assess competitor pric...,a market research analyst
2,Consumer Behavior,What factors influence Gen Z consumers when ch...,Gen Z consumers are influenced by mobile app u...,Consumers are more likely to be able to use th...
3,Market Trends,What trends are affecting consumer demand for ...,"Demand is being shaped by health awareness, re...",Consumers are increasingly relying on wearable...


In [43]:
fine_tuned_results = []

training_model.eval()

for _, row in test_df.iterrows():
    prompt_text = (
        "You are a market research analyst. "
        "Provide a clear, relevant, and actionable market research response.\n\n"
        f"Question: {row['prompt']}\n"
        "Answer:"
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    outputs = training_model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    fine_tuned_results.append({
        "category": row["category"],
        "prompt": row["prompt"],
        "expected_response": row["response"],
        "fine_tuned_response": response
    })

fine_tuned_outputs_df = pd.DataFrame(fine_tuned_results)

fine_tuned_outputs_df.to_csv(
    "fine_tuned_outputs.csv",
    index=False
)

display(fine_tuned_outputs_df)

,category,prompt,expected_response,fine_tuned_response
0,Regional Analysis,How should a company evaluate regional differe...,The company should analyze local income levels...,Identify regional differences.
1,Competitor Analysis,What factors should be included in a competito...,A clothing brand should assess competitor pric...,a market research analyst
2,Consumer Behavior,What factors influence Gen Z consumers when ch...,Gen Z consumers are influenced by mobile app u...,Consumers are more likely to be able to use th...
3,Market Trends,What trends are affecting consumer demand for ...,"Demand is being shaped by health awareness, re...",Consumers are increasingly relying on wearable...


In [44]:
comparison_df = pd.DataFrame({
    "category": test_df["category"].values,
    "prompt": test_df["prompt"].values,
    "expected_response": test_df["response"].values,
    "base_model_response": base_outputs_df["base_model_response"].values,
    "fine_tuned_response": fine_tuned_outputs_df["fine_tuned_response"].values
})

comparison_df.to_csv(
    "model_comparison.csv",
    index=False
)

display(comparison_df)

,category,prompt,expected_response,base_model_response,fine_tuned_response
0,Regional Analysis,How should a company evaluate regional differe...,The company should analyze local income levels...,Identify regional differences.,Identify regional differences.
1,Competitor Analysis,What factors should be included in a competito...,A clothing brand should assess competitor pric...,a market research analyst,a market research analyst
2,Consumer Behavior,What factors influence Gen Z consumers when ch...,Gen Z consumers are influenced by mobile app u...,Consumers are more likely to be able to use th...,Consumers are more likely to be able to use th...
3,Market Trends,What trends are affecting consumer demand for ...,"Demand is being shaped by health awareness, re...",Consumers are increasingly relying on wearable...,Consumers are increasingly relying on wearable...


In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def similarity_score(reference, candidate):
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([reference, candidate])
    return cosine_similarity(vectors[0:1], vectors[1:2])[0][0]

base_scores = []
fine_tuned_scores = []

for _, row in comparison_df.iterrows():

    base_score = similarity_score(
        row["expected_response"],
        row["base_model_response"]
    )

    fine_score = similarity_score(
        row["expected_response"],
        row["fine_tuned_response"]
    )

    base_scores.append(base_score)
    fine_tuned_scores.append(fine_score)

comparison_df["base_similarity"] = base_scores
comparison_df["fine_tuned_similarity"] = fine_tuned_scores

comparison_df.to_csv(
    "model_comparison_with_scores.csv",
    index=False
)

display(comparison_df)

,category,prompt,expected_response,base_model_response,fine_tuned_response,base_similarity,fine_tuned_similarity
0,Regional Analysis,How should a company evaluate regional differe...,The company should analyze local income levels...,Identify regional differences.,Identify regional differences.,0.000000,0.000000
1,Competitor Analysis,What factors should be included in a competito...,A clothing brand should assess competitor pric...,a market research analyst,a market research analyst,0.060972,0.060972
2,Consumer Behavior,What factors influence Gen Z consumers when ch...,Gen Z consumers are influenced by mobile app u...,Consumers are more likely to be able to use th...,Consumers are more likely to be able to use th...,0.109930,0.109930
3,Market Trends,What trends are affecting consumer demand for ...,"Demand is being shaped by health awareness, re...",Consumers are increasingly relying on wearable...,Consumers are increasingly relying on wearable...,0.000000,0.000000


In [46]:
print(
    "Average Base Model Similarity:",
    comparison_df["base_similarity"].mean()
)

print(
    "Average Fine-Tuned Model Similarity:",
    comparison_df["fine_tuned_similarity"].mean()
)

Average Base Model Similarity: 0.042725514781110786
Average Fine-Tuned Model Similarity: 0.042725514781110786


In [47]:
results_summary = pd.DataFrame({
    "Metric": [
        "Base Model Average Similarity",
        "Fine-Tuned Model Average Similarity",
        "Final Validation Loss"
    ],
    "Value": [
        comparison_df["base_similarity"].mean(),
        comparison_df["fine_tuned_similarity"].mean(),
        final_eval["eval_loss"]
    ]
})

results_summary.to_csv(
    "results_summary.csv",
    index=False
)

display(results_summary)

,Metric,Value
0,Base Model Average Similarity,0.042726
1,Fine-Tuned Model Average Similarity,0.042726
2,Final Validation Loss,NaN


In [48]:
!zip -r sba928_market_research_adapter.zip sba928_market_research_adapter

  adding: sba928_market_research_adapter/ (stored 0%)
  adding: sba928_market_research_adapter/tokenizer_config.json (deflated 83%)
  adding: sba928_market_research_adapter/README.md (deflated 66%)
  adding: sba928_market_research_adapter/adapter_config.json (deflated 59%)
  adding: sba928_market_research_adapter/tokenizer.json (deflated 75%)
  adding: sba928_market_research_adapter/adapter_model.safetensors (deflated 47%)


In [49]:
sba928_market_research_adapter.zip

NameError: name 'sba928_market_research_adapter' is not defined

In [50]:
!zip -r sba928_market_research_adapter.zip sba928_market_research_adapter

updating: sba928_market_research_adapter/ (stored 0%)
updating: sba928_market_research_adapter/tokenizer_config.json (deflated 83%)
updating: sba928_market_research_adapter/README.md (deflated 66%)
updating: sba928_market_research_adapter/adapter_config.json (deflated 59%)
updating: sba928_market_research_adapter/tokenizer.json (deflated 75%)
updating: sba928_market_research_adapter/adapter_model.safetensors (deflated 47%)
